# 04 — Inference & Kaggle Submission
**SF Crime Classification | Step 4 of 4**

> Prerequisite: Run `03_model_training.ipynb` first.

In [ ]:
%pip install xgboost lightgbm -q

In [ ]:
import pandas as pd
import numpy as np
import joblib

CATALOG     = "workspace"
SCHEMA      = "default"
VOLUME      = "sf_crime_data"
VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"

FEATURES = [
    "Hour","Minute","Month","Year","DayOfMonth","DayOfYear","WeekOfYear","Quarter","Season",
    "IsWeekend","IsNight","IsLateNight","IsMorning","IsAfternoon","IsEvening","IsRushHour","IsBusinessHours",
    "HourSin","HourCos","MonthSin","MonthCos","DowSin","DowCos","DaySin","DayCos",
    "X","Y","DistCenter","CoordFreq","X_round2","Y_round2","X_round3","Y_round3","GridX","GridY",
    "IsBlock","IsIntersection","StreetNum","StreetNumBin",
    "District_enc","DayOfWeek_enc","DistrictRate",
    "District_Hour","District_DayOfWeek","Hour_DayOfWeek","District_IsNight","District_IsWeekend",
]
print("✅ Config ready")

In [ ]:
# ── Load model from Volume ────────────────────────────────
bundle     = joblib.load(f"{VOLUME_PATH}/best_model.joblib")
classifier = bundle["model"]
best_name  = bundle["name"]
print(f"✅ Loaded: {best_name}")
print(f"   ROC-AUC weighted: {bundle['metrics']['roc_auc_weighted']:.4f}")

In [ ]:
# ── Load test features & label map ───────────────────────
test_pdf = spark.read.table(f"workspace.default.features_test").toPandas()
class_names = (spark.read.table(f"workspace.default.label_map")
    .orderBy("label_idx").toPandas()["category"].tolist())

print(f"Test rows    : {len(test_pdf):,}")
print(f"Crime classes: {len(class_names)}")

In [ ]:
# ── Batched inference ─────────────────────────────────────
BATCH_SIZE = 50_000
X_test     = test_pdf[FEATURES].values
all_probs  = []

for start in range(0, len(X_test), BATCH_SIZE):
    batch = X_test[start:start+BATCH_SIZE]
    all_probs.append(classifier.predict_proba(batch))
    print(f"  {min(start+BATCH_SIZE, len(X_test)):,} / {len(X_test):,}")

probs_all = np.vstack(all_probs)
print(f"\n✅ Inference complete — shape: {probs_all.shape}")

In [ ]:
submission = pd.DataFrame(probs_all, columns=class_names)
submission.insert(0, "Id", test_pdf["Id"].values)
assert abs(submission[class_names].iloc[0].sum() - 1.0) < 1e-4
print(f"✅ {len(submission):,} rows, {len(class_names)} classes")
submission.head(3)

In [ ]:
OUT_PATH = f"/Volumes/workspace/default/sf_crime_data/submission.csv"
submission.to_csv(OUT_PATH, index=False)
print(f"✅ Saved: {OUT_PATH}")
print("\nDownload: Catalog > workspace > default > Volumes > sf_crime_data > submission.csv > ⋮ > Download")

---
## Pipeline Complete ✅